In [3]:
#Libraries
from sqlalchemy import create_engine
import pyodbc
import pandas as pd
import numpy as np
import glob
import os 
import urllib

In [4]:
pip install pyodbc

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [11]:
os.getcwd()

'D:\\Data Analyst\\EDUFIN\\csv files'

In [9]:
os.chdir(
'D:\\Data Analyst\\EDUFIN\\csv files')

In [15]:
glob.glob(
    'D:\\Data Analyst\\EDUFIN\\csv files/loans.csv'
)

['D:\\Data Analyst\\EDUFIN\\csv files/loans.csv']

In [16]:

all_files = glob.glob('D:\\Data Analyst\\EDUFIN\\csv files/loans.csv')
dfs = []
for file in all_files:
    df1 = pd.read_csv(file)

In [17]:
df1.head()

,loan_id,customer_id,institution_id,loan_amount,loan_status,interest_rate,loan_tenure_months,application_date,disbursement_date,maturity_date,emi_amount,purpose_of_loan
0,1,2440,2954,190607.1250,Defaulted,11.39,84,08-12-2021,23-12-2021,16-11-2028,3302.540039,Living Expenses
1,2,2440,4741,425798.3750,Active,14.44,48,01-01-2022,11-01-2022,21-12-2025,11728.730470,Course Fees + Living
2,3,2440,902,318341.2500,Defaulted,11.65,96,06-03-2023,12-04-2023,01-03-2031,5112.870117,Course Fees + Living
3,4,4195,4724,327505.7500,Active,10.19,48,20-12-2024,31-01-2025,10-01-2029,8336.240234,Course Fees + Living
4,5,4195,244,641567.8125,Active,11.69,84,16-11-2023,10-12-2023,03-11-2030,11219.620120,Hostel Fees


In [18]:
#Change Numeric Column Values to Integer
for col in df1.columns:
    if pd.api.types.is_numeric_dtype(df1[col]):
        # If all non‑missing values are whole numbers, cast to Int64
        if (df1[col].dropna() % 1 == 0).all():
            df1[col] = df1[col].astype("Int64")   # nullable integer
        elif (df1[col].dropna() % 1 != 0).any():
            # Otherwise keep as float (decimals preserved)
             df1[col] = df1[col].astype("float")
        else:
            df1[col] = df1[col].astype("str")
    else:
        #Everything else (text, regions, etc)
        df1[col] = df1[col].astype("str")

In [19]:
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   loan_id             5000 non-null   Int64  
 1   customer_id         5000 non-null   Int64  
 2   institution_id      5000 non-null   Int64  
 3   loan_amount         5000 non-null   float64
 4   loan_status         5000 non-null   str    
 5   interest_rate       5000 non-null   float64
 6   loan_tenure_months  5000 non-null   Int64  
 7   application_date    5000 non-null   str    
 8   disbursement_date   5000 non-null   str    
 9   maturity_date       5000 non-null   str    
 10  emi_amount          5000 non-null   float64
 11  purpose_of_loan     5000 non-null   str    
dtypes: Int64(4), float64(3), str(5)
memory usage: 488.4 KB


In [20]:
#Step 1: Define the Server instance and Database Name
server = "DESKTOP-U507UFU\\MSSQLSERVER01"
database = "Practice1"
#Step2: Instatiate a connection String 
conn_str = (
    "Driver={ODBC Driver 17 for SQL Server};"
    f"Server={server};"
    f"Database={database};"
    "Trusted_Connection=yes;"
)
#Step3: Connect to SQL database and Append raw files to the respective in form tables 
params = urllib.parse.quote_plus(conn_str)
engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}",
    fast_executemany=True 
                    
                      )

df1.to_sql(
    "loans",
    engine,
    if_exists="append",
    index=False,
    chunksize=10000
)

-1